In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.layers import LSTM, Dense
from kerastuner.tuners import RandomSearch

from sklearn.model_selection import GridSearchCV

import warnings
warnings.filterwarnings('ignore')

import time

2024-04-05 10:46:26.402991: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-05 10:46:27.195519: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/tmp/ipykernel_22189/4096088174.py:13: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import RandomSearch


In [2]:
%store -r FD1_X_Train
%store -r FD1_X_Test

%store -r FD1_y_Train
%store -r FD1_y_Test


FD1_X_Train_array = FD1_X_Train.values
FD1_X_Test_array = FD1_X_Test.values
FD1_y_Train_array = FD1_y_Train.values
FD1_y_Test_array = FD1_y_Test.values

X_train, X_val, y_train, y_val = train_test_split(FD1_X_Train, FD1_y_Train, test_size=0.2, random_state=42)

print("Training set - X:", X_train.shape, " y:", y_train.shape)
print("Validation set - X:", X_val.shape, " y:", y_val.shape)

Training set - X: (16504, 15)  y: (16504,)
Validation set - X: (4127, 15)  y: (4127,)


### Long-Short Term Memory

In [3]:
def create_lstm_model(input_shape):
    model = Sequential([
        layers.Reshape((input_shape[0], 1), input_shape=input_shape),
        layers.LSTM(128, activation="relu", return_sequences=True),
        layers.LSTM(64, activation="relu", return_sequences=True),
        layers.LSTM(32, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(1)
    ])
    model.compile(loss="mse", optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))
    return model

In [4]:
def scheduler(epoch):
    if epoch < 5:
        return 0.001
    else:
        return 0.0001

callback = tf.keras.callbacks.LearningRateScheduler(scheduler, verbose = 1)

FD1_X_Train_array = FD1_X_Train.values
FD1_X_Test_array = FD1_X_Test.values

input_shape = FD1_X_Train_array.shape[1:]

lstm_model = create_lstm_model(input_shape)

lstm_model.compile(loss='mse', optimizer='adam')

FD1_y_Train_array = FD1_y_Train_array.astype('float32')
FD1_y_Test_array = FD1_y_Test_array.astype('float32')

history = lstm_model.fit(FD1_X_Train_array, FD1_y_Train_array, epochs=10, batch_size=128, callbacks=callback, validation_data=(FD1_X_Test_array, FD1_y_Test_array))

2024-04-05 10:46:28.007761: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355



Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/10


2024-04-05 10:46:28.054377: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


162/162 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - loss: 8266.2979 - val_loss: 2301.1404 - learning_rate: 0.0010

Epoch 2: LearningRateScheduler setting learning rate to 0.001.
Epoch 2/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 1893.7338 - val_loss: 2773.3735 - learning_rate: 0.0010

Epoch 3: LearningRateScheduler setting learning rate to 0.001.
Epoch 3/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 1913.3591 - val_loss: 1622.2133 - learning_rate: 0.0010

Epoch 4: LearningRateScheduler setting learning rate to 0.001.
Epoch 4/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 1171.8754 - val_loss: 999.3173 - learning_rate: 0.0010

Epoch 5: LearningRateScheduler setting learning rate to 0.001.
Epoch 5/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 752.4960 - val_loss: 1026.1442 - learning_rate: 0.0010

Epoch 6: LearningRateScheduler setting learning rate to 0.0001.
Epoch 6/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 673.0407 - val_loss: 845.2060 - learning_rate:

In [20]:
predictions = lstm_model.predict(FD1_X_Test)

mse = mean_squared_error(FD1_y_Test, predictions)
mae = mean_absolute_error(FD1_y_Test, predictions)
rmse = mean_squared_error(FD1_y_Test, predictions, squared=False)
r2 = r2_score(FD1_y_Test, predictions)

print("Mean Squared Error (MSE):", mse)
print("Mean Absolute Error (MAE):", mae)
print("Root Mean Squared Error (RMSE):", rmse)
print("R-squared (R2) Score:", r2)

%store mse
%store rmse
%store mae
%store r2

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Mean Squared Error (MSE): 1034.498375345838
Mean Absolute Error (MAE): 26.807604808807373
Root Mean Squared Error (RMSE): 32.163618816075996
R-squared (R2) Score: 0.40094007367676276
Stored 'mse' (float64)
Stored 'rmse' (float64)
Stored 'mae' (float64)
Stored 'r2' (float64)
